# Video DiT Training Notebook

This notebook provides an easy interface to train the Video DiT model. You can run this on Google Colab or any Jupyter environment.

## Setup

First, let's install the required dependencies and clone/setup the repository.

In [1]:
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install diffusers transformers accelerate
!pip install tqdm imageio opencv-python
!pip install torchmetrics lpips

Looking in indexes: https://download.pytorch.org/whl/cu121
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 2.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 14.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 37.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 96.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 13.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 2.5 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 13.7 MB/s eta 0:00:000

In [ ]:
# Clone or setup the repository
# If running on Colab, clone the repo
import os
if not os.path.exists('video-generation-model'):
    !git clone https://github.com/your-username/video-generation-model.git
    %cd video-generation-model
else:
    %cd video-generation-model

Cloning into 'video-generation-model'...
Username for 'https://github.com': 

## Configuration

Set up your training parameters below. You can modify these values to customize your training run.

In [ ]:
# Training Configuration
config = {
    # Data paths
    'latent_dir': '../latents',  # Path to latent directory
    'vae_checkpoint': '../vae_checkpoint_epoch_24.pth',  # VAE checkpoint path
    
    # Training hyperparameters
    'batch_size': 4,
    'lr': 1e-4,
    'epochs': 100,
    
    # Model architecture
    'dim': 768,
    'depth': 8,
    'heads': 8,
    
    # Training settings
    'save_every': 10,
    'generate_every': 5,
    'amp': True,  # Use automatic mixed precision
    
    # Data settings
    'num_workers': 2,
    'timesteps': 1000,
    'in_channels': 4,
    'T': 16,
    'H': 32,
    'W': 32
}

print("Training configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## Data Preparation

Make sure your latent data is available. If you're using Google Colab, you'll need to upload or mount your data.

In [ ]:
# Check if latent data exists
import os
if os.path.exists(config['latent_dir']):
    latent_files = [f for f in os.listdir(config['latent_dir']) if f.endswith('.pt')]
    print(f"Found {len(latent_files)} latent files in {config['latent_dir']}")
    
    # Show a few examples
    if latent_files:
        print("Sample files:", latent_files[:5])
else:
    print(f"Warning: Latent directory {config['latent_dir']} not found!")
    print("Please ensure your latent data is available.")
    print("You can:")
    print("1. Upload latents to Colab")
    print("2. Mount Google Drive")
    print("3. Generate latents first using generate_latents.py")

## Start Training

Now let's start the training process. This will take some time depending on your configuration.

In [ ]:
# Build the training command
cmd_parts = [
    'cd model',
    'python train_video_dit.py',
    f"--latent_dir {config['latent_dir']}",
    f"--vae_checkpoint {config['vae_checkpoint']}",
    f"--batch_size {config['batch_size']}",
    f"--lr {config['lr']}",
    f"--dim {config['dim']}",
    f"--depth {config['depth']}",
    f"--epochs {config['epochs']}",
    f"--save_every {config['save_every']}",
    f"--generate_every {config['generate_every']}",
    f"--heads {config['heads']}",
    f"--timesteps {config['timesteps']}",
    f"--in_channels {config['in_channels']}",
    f"--T {config['T']}",
    f"--H {config['H']}",
    f"--W {config['W']}",
    f"--num_workers {config['num_workers']}"
]

if config['amp']:
    cmd_parts.append('--amp')

training_command = ' \
'.join(cmd_parts)
print("Training command:")
print(training_command)

In [ ]:
# Execute training
# Note: This will run for a long time. Consider using Colab's GPU runtime
!{training_command}

## Alternative: Quick Training Command

If you prefer to run the exact command from the terminal, use this:

In [ ]:
# Quick training command (copy and run in terminal)
quick_command = f"""
cd model
python train_video_dit.py \
  --latent_dir {config['latent_dir']} \
  --vae_checkpoint {config['vae_checkpoint']} \
  --amp \
  --batch_size {config['batch_size']} \
  --lr {config['lr']} \
  --dim {config['dim']} \
  --depth {config['depth']} \
  --epochs {config['epochs']} \
  --save_every {config['save_every']} \
  --generate_every {config['generate_every']}
"""

print("Quick training command:")
print(quick_command)

## Monitoring Training

The training will save checkpoints every `save_every` epochs. You can monitor the progress by:

1. Checking the loss values printed during training
2. Loading saved checkpoints to generate sample videos
3. Using tensorboard or similar tools if you set them up

## Results

After training completes, your model checkpoints will be saved in the project root directory with names like `video_dit_checkpoint_epoch_X.pth`.